# Jupyter Notebook for Calculating Statistics

**NOTE**: the `json` files that contain overall metrics and also the linear regression model results were created manually.
This can be done with some Python code very easily. However, the code for this is not included in this notebook.

In [2]:
import pandas as pd
import numpy as np
from math import sqrt
import altair as alt
from utils import save_dataset_to_csv
import statsmodels.formula.api as smf
from tqdm import tqdm
import tiktoken

In [3]:
model_metadata = {
    "alpacare-7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "biomedgpt7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "biomistral7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "claude_3.5-haiku": {"model_type": "generalist closed", "model_size_in_b": None},
    "claude_3.5-sonnet": {"model_type": "generalist closed", "model_size_in_b": None},
    "gemini_1.5_flash": {"model_type": "generalist closed", "model_size_in_b": None},
    "gemini_1.5_flash-8B": {"model_type": "generalist closed", "model_size_in_b": 8},
    "gpt4o": {"model_type": "generalist closed", "model_size_in_b": None},
    "gpt4o-mini": {"model_type": "generalist closed", "model_size_in_b": None},
    "gpt35": {"model_type": "generalist closed", "model_size_in_b": 175},
    "llama2_chat-7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "llama2_chat-13B": {"model_type": "generalist open", "model_size_in_b": 13},
    "llama2_chat-70B": {"model_type": "generalist open", "model_size_in_b": 70},
    "llama3_instruct-8B": {"model_type": "generalist open", "model_size_in_b": 8},
    "llama3_instruct-70B": {"model_type": "generalist open", "model_size_in_b": 70},
    "med42-8B": {"model_type": "biomedical open", "model_size_in_b": 8},
    "med42-70B": {"model_type": "biomedical open", "model_size_in_b": 70},
    "mistral_instruct7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "olmo2_instruct-7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "olmo2_instruct-13B": {"model_type": "generalist open", "model_size_in_b": 13},
    "openbiollm-8B": {"model_type": "biomedical open", "model_size_in_b": 8},
    "openbiollm-70B": {"model_type": "biomedical open", "model_size_in_b": 70}
}

## Spin Detection Task

In [4]:
# detection_overall_metrics.json was manually created
detection_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/detection_overall_metrics_all_new.json", orient="index")

detection_stats_df["model_name"] = detection_stats_df.index
detection_stats_df["model_type"] = detection_stats_df.index.map(lambda x: model_metadata[x]["model_type"])
detection_stats_df["model_size_in_b"] = detection_stats_df.index.map(lambda x: model_metadata[x]["model_size_in_b"])
# remove index
detection_stats_df.reset_index(drop=True, inplace=True)

print(f"Number of models: {len(detection_stats_df)}")

detection_stats_df.sort_index(inplace=True) # alphabetical order
detection_stats_df

Number of models: 22


,accuracy,precision,recall,f1,model_name,model_type,model_size_in_b
0,0.536667,0.537931,0.520000,0.528814,olmo2_instruct-7B,generalist open,7.0
1,0.536667,1.000000,0.073333,0.136646,olmo2_instruct-13B,generalist open,13.0
2,0.500000,0.000000,0.000000,0.000000,mistral_instruct7B,generalist open,7.0
3,0.530000,1.000000,0.060000,0.113208,med42-8B,biomedical open,8.0
4,0.510000,1.000000,0.020000,0.039216,biomistral7B,biomedical open,7.0
5,0.678571,0.678571,1.000000,0.808511,openbiollm-8B,biomedical open,8.0
6,0.500000,0.500000,1.000000,0.666667,llama2_chat-7B,generalist open,7.0
7,0.680000,0.631068,0.866667,0.730337,llama2_chat-13B,generalist open,13.0
8,0.643333,1.000000,0.286667,0.445596,llama3_instruct-8B,generalist open,8.0
9,0.626667,0.593137,0.806667,0.683616,llama2_chat-70B,generalist open,70.0


### Average of accuracy by model type

In [5]:
# Group by model type and calculate mean accuracy and standard deviation
accuracy_by_model_type = detection_stats_df.groupby('model_type')['accuracy'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
accuracy_by_model_type.columns = ['model_type', 'mean_accuracy', 'std_deviation']

print(accuracy_by_model_type)

          model_type  mean_accuracy  std_deviation
0    biomedical open       0.555327       0.075280
1  generalist closed       0.672381       0.078733
2    generalist open       0.579167       0.069322


In [6]:
# Group by model type and calculate mean precision and standard deviation
precision_by_model_type = detection_stats_df.groupby('model_type')['precision'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
precision_by_model_type.columns = ['model_type', 'mean_precision', 'std_deviation']

print(precision_by_model_type)

          model_type  mean_precision  std_deviation
0    biomedical open        0.761650       0.221866
1  generalist closed        0.766933       0.136131
2    generalist open        0.657767       0.343962


In [7]:
# Group by model type and calculate mean recall and standard deviation
recall_by_model_type = detection_stats_df.groupby('model_type')['recall'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
recall_by_model_type.columns = ['model_type', 'mean_recall', 'std_deviation']

print(recall_by_model_type)

          model_type  mean_recall  std_deviation
0    biomedical open     0.509640       0.459681
1  generalist closed     0.605714       0.305249
2    generalist open     0.471667       0.383298


In [8]:
# Group by model type and calculate mean f1 and standard deviation
f1_by_model_type = detection_stats_df.groupby('model_type')['f1'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
f1_by_model_type.columns = ['model_type', 'mean_f1', 'std_deviation']

print(f1_by_model_type)

          model_type   mean_f1  std_deviation
0    biomedical open  0.433655       0.291180
1  generalist closed  0.600528       0.248930
2    generalist open  0.444041       0.265849


#### Plots

In [9]:
# Create a dictionary for custom labels
custom_labels = {
    "alpacare-7B": "AlpaCare 7B",
    "biomedgpt7B": "BioMedGPT 7B",
    "biomistral7B": "BioMistral 7B",
    "claude_3.5-haiku": "Claude3.5 Haiku",
    "claude_3.5-sonnet": "Claude3.5 Sonnet", 
    "gemini_1.5_flash": "Gemini1.5 Flash",
    "gemini_1.5_flash-8B": "Gemini1.5 Flash 8B",
    "gpt4o": "GPT-4o",
    "gpt4o-mini": "GPT-4o Mini",
    "gpt35": "GPT-3.5",
    "llama2_chat-7B": "Llama2 Chat 7B",
    "llama2_chat-13B": "Llama2 Chat 13B",
    "llama2_chat-70B": "Llama2 Chat 70B",
    "llama3_instruct-8B": "Llama3 Instruct 8B",
    "llama3_instruct-70B": "Llama3 Instruct 70B",
    "med42-8B": "Med42 8B",
    "med42-70B": "Med42 70B",
    "mistral_instruct7B": "Mistral Instruct 7B",
    "olmo2_instruct-7B": "Olmo2 Instruct 7B",
    "olmo2_instruct-13B": "Olmo2 Instruct 13B",
    "openbiollm-8B": "OpenBioLLM 8B",
    "openbiollm-70B": "OpenBioLLM 70B"
}

detection_stats_df['model_name_custom'] = detection_stats_df['model_name'].map(custom_labels)

color_mapping = {
    'biomedical open': '#0868ac', 
    'generalist closed': '#7bccc4',
    'generalist open': '#bae4bc',
}

# Create the bar chart
chart = alt.Chart(detection_stats_df).mark_bar().encode(
    y=alt.Y('model_name_custom:N', sort='-x', title='Model Name'),
    x=alt.X('accuracy:Q', title='Accuracy'),
    color=alt.Color('model_type:N', title='Model Type',
                    scale=alt.Scale(domain=list(color_mapping.keys()), range=list(color_mapping.values())),
                    legend=alt.Legend(
                    orient='none',
                    legendX=130, legendY=-45,
                    direction='horizontal',
                    titleAnchor='middle'))  # Legend at the bottom
).properties(
    width=800,
)

# Add value labels with increased font size
text = chart.mark_text(
    align='center',
    baseline='middle',
    fontWeight='bold',
    dx=20  # Adjust the position of the text
).encode(
    text=alt.Text('accuracy:Q', format='.2f'),
    color=alt.value('black'),
)

# Add a mean rule
avg_rule = alt.Chart(detection_stats_df).mark_rule(color='red').encode(
    x='mean(accuracy):Q',
    size=alt.value(2)
)

# Add a 50% chance rule
chance_rule = alt.Chart(detection_stats_df).mark_rule(color='gray').encode(
    x='min(accuracy):Q',
    size=alt.value(2),
    strokeDash=alt.value([10, 10])
)

# Increase font size for axis labels, titles, and other components
chart_config = {
    "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
    "header": {"labelFontSize": 20, "titleFontSize": 22},  # Title and facet headers (if any)
    "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
    "text": {"fontSize": 20},  # Text mark size
}

# Combine chart and text, and apply the config
c_t = chart + avg_rule + chance_rule + text
c_t = c_t.configure(**chart_config)  # Apply the global configuration

# Save to HTML
c_t.save("./plots/detection_accuracy_by_model.html")

# Display the chart
c_t

alt.LayerChart(...)

In [9]:
# # Create the bar chart
# chart = alt.Chart(detection_stats_df).mark_bar().encode(
#     y=alt.Y('model_name_custom:N', sort='-x', title='Model Name'),
#     x=alt.X('accuracy:Q', title='Accuracy'),
#    color=alt.Color('model_type:N', title='Model Type', legend=alt.Legend(
#         orient='none',
#         legendX=130, legendY=-45,
#         direction='horizontal',
#         titleAnchor='middle'), scale=alt.Scale(range=["#808080", "#A9A9A9", "#D3D3D3", "#BEBEBE"]))  # Legend at the bottom
# ).properties(
#     width=800,
# )

# # Add value labels with increased font size
# text = chart.mark_text(
#     align='center',
#     baseline='middle',
#     fontWeight='bold',
#     dx=18  # Adjust the position of the text
# ).encode(
#     text=alt.Text('accuracy:Q', format='.2f'),
#     color=alt.value('black')  # Set text color to black
# )

# # Add a mean rule
# rule = alt.Chart(detection_stats_df).mark_rule(color='gray').encode(
#     x='mean(accuracy):Q',
#     size=alt.value(2)
# )

# # Increase font size for axis labels, titles, and other components
# chart_config = {
#     "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
#     "header": {"labelFontSize": 20, "titleFontSize": 22},  # Title and facet headers (if any)
#     "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
#     "text": {"fontSize": 20},  # Text mark size
# }

# # Combine chart and text, and apply the config
# c_t = chart + rule + text
# c_t = c_t.configure(**chart_config)  # Apply the global configuration

# # Save to HTML
# c_t.save("./plots/detection_accuracy_by_model_gray.html")

# # Display the chart
# c_t

In [21]:
# Plot average accuracy by model_type and add error bars
bars = alt.Chart(detection_stats_df).mark_bar().encode(
    x=alt.X('model_type:N', title='Model Type', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('mean(accuracy):Q', title='Mean Accuracy'),
    color=alt.Color('model_type:N', title='Model Type', legend=None)
).properties(
    title='Average Accuracy by Model Type',
    width=800  # Set the width to 800 pixels
)

error_bars = alt.Chart(detection_stats_df).mark_errorbar(extent='stdev').encode(
    x=alt.X('model_type:N'),
    y=alt.Y('accuracy:Q')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    dy=-5  # Adjust the position of the text
).encode(
    text=alt.Text('mean(accuracy):Q', format='.2f')
)

plot_ = alt.layer(bars, error_bars, text)

plot_.save("./plots/average_accuracy_by_model_type.html")

plot_

alt.LayerChart(...)

### Average of accuracy by model size

In [10]:
# model size in buckets (0-10B, 11-20B, 22-100B, 100B+/NaN)
def model_size_bucket(model_size): 
    if model_size is None or pd.isna(model_size):
        return "Unknown"
    elif model_size >= 100:
        return "100B+"
    elif model_size <= 10:
        return "0-10B"
    elif model_size <= 20:
        return "11-20B"
    else:
        return "21-100B"

In [11]:
# average accuracy by model size
detection_stats_df["model_size_bucket"] = detection_stats_df["model_size_in_b"].map(model_size_bucket)

accuracy_by_model_size = detection_stats_df.groupby('model_size_bucket')['accuracy'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
accuracy_by_model_size.columns = ['model_size_bucket', 'mean_accuracy', 'std_deviation']

print(accuracy_by_model_size)

  model_size_bucket  mean_accuracy  std_deviation
0             0-10B       0.555396       0.082589
1             100B+       0.513333            NaN
2            11-20B       0.608333       0.101352
3           21-100B       0.610000       0.019052
4           Unknown       0.700667       0.043551


In [12]:
# average precision by model size
precision_by_model_size = detection_stats_df.groupby('model_size_bucket')['precision'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
precision_by_model_size.columns = ['model_size_bucket', 'mean_precision', 'std_deviation']

print(precision_by_model_size)

  model_size_bucket  mean_precision  std_deviation
0             0-10B        0.653686       0.320056
1             100B+        1.000000            NaN
2            11-20B        0.815534       0.260874
3           21-100B        0.819403       0.175398
4           Unknown        0.703336       0.085784


In [13]:
# average recall by model size
recall_by_model_size = detection_stats_df.groupby('model_size_bucket')['recall'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
recall_by_model_size.columns = ['model_size_bucket', 'mean_recall', 'std_deviation']

print(recall_by_model_size)

  model_size_bucket  mean_recall  std_deviation
0             0-10B     0.531415       0.432072
1             100B+     0.026667            NaN
2            11-20B     0.470000       0.560971
3           21-100B     0.386667       0.286951
4           Unknown     0.750667       0.156105


In [14]:
# average f1 score by model size 
f1_by_model_size_bucket = detection_stats_df.groupby('model_size_bucket')['f1'].agg(['mean', 'std']).reset_index()

# Rename columns for clarity
f1_by_model_size_bucket.columns = ['model_size_bucket', 'mean_f1', 'std_deviation']

print(f1_by_model_size_bucket)

  model_size_bucket   mean_f1  std_deviation
0             0-10B  0.449738       0.292103
1             100B+  0.051948            NaN
2            11-20B  0.433492       0.419803
3           21-100B  0.455238       0.165850
4           Unknown  0.710870       0.049923


#### Plots

In [22]:
bars = alt.Chart(detection_stats_df).mark_bar().encode(
    x=alt.X('model_name:N', sort='-y', title='Model Name'),
    y=alt.Y('accuracy:Q', title='Accuracy'),
    color=alt.Color('model_size_bucket:N', title='Model Size Bucket')
).properties(
    title='Accuracy by Model Size Bucket',
    width=800,
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    dy=-5  # Adjust the position of the text
).encode(
    text=alt.Text('accuracy:Q', format='.2f')
)

plot_ = bars + text

plot_.save("./plots/average_accuracy_by_model_size.html")

plot_

alt.LayerChart(...)

In [23]:
# Plot average accuracy by model_size_bucket and add error bars
bars = alt.Chart(detection_stats_df).mark_bar().encode(
    x=alt.X('model_size_bucket:N', title='Model Size Bucket', axis=alt.Axis(labelAngle=0), sort=['0-10B', '11-20B', '21-100B', '100B+', 'Unknown']),
    y=alt.Y('mean(accuracy):Q', title='Mean Accuracy'),
    color=alt.Color('model_size_bucket:N', title='Model Size Bucket', legend=None)
).properties(
    title='Average Accuracy by Model Size',
    width=800  # Set the width to 800 pixels
)

error_bars = alt.Chart(detection_stats_df).mark_errorbar(extent='stdev').encode(
    x=alt.X('model_size_bucket:N', sort=['0-10B', '11-20B', '21-100B', '100B+', 'Unknown']),
    y=alt.Y('accuracy:Q')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    dy=-5  # Adjust the position of the text
).encode(
    text=alt.Text('mean(accuracy):Q', format='.2f')
)

plot_ = alt.layer(bars, error_bars, text)

plot_.save("./plots/average_accuracy_by_model_size_bucket.html")

plot_

alt.LayerChart(...)

In [24]:
# scatter plot of model size vs accuracy with model names as labels
scatter_plot = alt.Chart(detection_stats_df).mark_circle().encode(
    x=alt.X('model_size_in_b:Q', title='Model Size (in Billion Parameters)'),
    y=alt.Y('accuracy:Q', title='Accuracy'),
    color=alt.Color('model_type:N', title='Model Type')
).properties(
    title='Model Size vs Accuracy',
    width=800,  # Set the width to 800 pixels
    height=400  # Set the height to 400 pixels
)

text = scatter_plot.mark_text(
    align='left',
    baseline='middle',
    dx=7,  # Adjust the position of the text
    dy=-5,  # Adjust the vertical position of the text
).encode(
    text='model_name:N'
)

plot_ = scatter_plot + text

plot_.save("./plots/model_size_vs_accuracy.html")

plot_

alt.LayerChart(...)

## RCT Trial Result Interpretation Task

In [25]:
# interpretation_overall_metrics.json was manually created
interpretation_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/interpretation_overall_metrics_all_new.json", orient="index")

interpretation_stats_df["model_name"] = interpretation_stats_df.index
interpretation_stats_df["model_type"] = interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_type"])
interpretation_stats_df["model_size_in_b"] = interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_size_in_b"])
# remove index
interpretation_stats_df.reset_index(drop=True, inplace=True)

print(f"Number of models: {len(interpretation_stats_df)}")

interpretation_stats_df.sort_index(inplace=True) # alphabetical order
interpretation_stats_df

Number of models: 22


,benefit_answer_mean_diff,rigor_answer_mean_diff,importance_answer_mean_diff,full_text_answer_mean_diff,another_trial_answer_mean_diff,overall_mean_diff_avg,model_name,model_type,model_size_in_b
0,3.377778,0.420000,1.100000,2.160000,1.573333,1.726222,olmo2_instruct-7B,generalist open,7.0
1,2.946667,0.766667,1.753333,3.013333,2.440000,2.184000,olmo2_instruct-13B,generalist open,13.0
2,1.766667,0.046667,1.620000,1.580000,1.253333,1.253333,mistral_instruct7B,generalist open,7.0
3,2.226667,0.240000,0.933333,1.120000,1.653333,1.234667,med42-8B,biomedical open,8.0
4,0.626667,0.026667,0.786667,0.547619,0.496667,0.496857,biomistral7B,biomedical open,7.0
5,3.401575,-0.062500,2.900000,4.420635,0.909091,2.313760,openbiollm-8B,biomedical open,8.0
6,1.663194,0.160000,0.966667,0.260000,-0.035398,0.602893,llama2_chat-7B,generalist open,7.0
7,2.402027,0.300000,1.193333,1.600000,1.506667,1.400405,llama2_chat-13B,generalist open,13.0
8,1.686667,0.066667,0.673333,0.880000,1.253333,0.912000,llama3_instruct-8B,generalist open,8.0
9,1.672932,0.126667,0.586667,0.513333,0.398551,0.659630,llama2_chat-70B,generalist open,70.0


In [26]:
human_expert_stats = {
        "benefit_answer": {"mean_diff": 0.71, "ci_lower": 0.07, "ci_upper": 1.35},
        "rigor_answer": {"mean_diff": -0.59, "ci_lower": -1.13, "ci_upper": -0.05},
        "importance_answer": {"mean_diff": -0.38, "ci_lower": -0.95, "ci_upper": 0.19},
        "full_text_answer": {"mean_diff": 0.77, "ci_lower": 0.08, "ci_upper": 1.47},
        "another_trial_answer": {"mean_diff": 0.64, "ci_lower": -0.03, "ci_upper": 1.31}
    }

human_expert_stats_df = pd.DataFrame(human_expert_stats).T
human_expert_stats_df["metric"] = human_expert_stats_df.index
# remove index
human_expert_stats_df.reset_index(drop=True, inplace=True)
human_expert_stats_df["method"] = "human experts"

human_expert_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,0.71,0.07,1.35,benefit_answer,human experts
1,-0.59,-1.13,-0.05,rigor_answer,human experts
2,-0.38,-0.95,0.19,importance_answer,human experts
3,0.77,0.08,1.47,full_text_answer,human experts
4,0.64,-0.03,1.31,another_trial_answer,human experts


In [10]:
def calculate_confidence_interval(df, df_column_name):
    mean_diff = df[df_column_name].mean()  # Calculate the mean
    std_dev = df[df_column_name].std()  # Calculate the standard deviation
    n = len(df[df_column_name])  # Sample size

    # Calculate the margin of error for 95% CI (z = 1.96)
    z = 1.96
    margin_of_error = z * (std_dev / sqrt(n))

    # Calculate the 95% Confidence Interval
    ci_lower = mean_diff - margin_of_error
    ci_upper = mean_diff + margin_of_error

    return ci_lower, ci_upper

In [11]:
def calculate_model_stats(interpretation_stats_df, method_name = "all LLMs"):
    # calculate the average of all model metrics and calculate 95% CI
    average_model_benefit = interpretation_stats_df["benefit_answer_mean_diff"].mean()
    ci_lower_model_benefit, ci_upper_model_benefit = calculate_confidence_interval(interpretation_stats_df, "benefit_answer_mean_diff")

    average_model_rigor = interpretation_stats_df["rigor_answer_mean_diff"].mean()
    ci_lower_model_rigor, ci_upper_model_rigor = calculate_confidence_interval(interpretation_stats_df, "rigor_answer_mean_diff")

    average_model_importance = interpretation_stats_df["importance_answer_mean_diff"].mean()
    ci_lower_model_importance, ci_upper_model_importance = calculate_confidence_interval(interpretation_stats_df, "importance_answer_mean_diff")

    average_model_full_text = interpretation_stats_df["full_text_answer_mean_diff"].mean()
    ci_lower_model_full_text, ci_upper_model_full_text = calculate_confidence_interval(interpretation_stats_df, "full_text_answer_mean_diff")

    average_model_another_trial = interpretation_stats_df["another_trial_answer_mean_diff"].mean()
    ci_lower_model_another_trial, ci_upper_model_another_trial = calculate_confidence_interval(interpretation_stats_df, "another_trial_answer_mean_diff")

    model_stats = {
        "benefit_answer": {"mean_diff": average_model_benefit, "ci_lower": ci_lower_model_benefit, "ci_upper": ci_upper_model_benefit},
        "rigor_answer": {"mean_diff": average_model_rigor, "ci_lower": ci_lower_model_rigor, "ci_upper": ci_upper_model_rigor},
        "importance_answer": {"mean_diff": average_model_importance, "ci_lower": ci_lower_model_importance, "ci_upper": ci_upper_model_importance},
        "full_text_answer": {"mean_diff": average_model_full_text, "ci_lower": ci_lower_model_full_text, "ci_upper": ci_upper_model_full_text},
        "another_trial_answer": {"mean_diff": average_model_another_trial, "ci_lower": ci_lower_model_another_trial, "ci_upper": ci_upper_model_another_trial}
    }

    model_stats_df = pd.DataFrame(model_stats).T
    model_stats_df["metric"] = model_stats_df.index
    # remove index
    model_stats_df.reset_index(drop=True, inplace=True)
    model_stats_df["method"] = method_name

    return model_stats_df

In [29]:
# calculate the average of all model metrics and calculate 95% CI
model_stats_df = calculate_model_stats(interpretation_stats_df)

model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.991332,1.567488,2.415176,benefit_answer,all LLMs
1,0.134005,0.032919,0.235091,rigor_answer,all LLMs
2,0.782012,0.387193,1.176831,importance_answer,all LLMs
3,1.471335,1.026661,1.916010,full_text_answer,all LLMs
4,1.345742,1.051612,1.639871,another_trial_answer,all LLMs


In [30]:
# get average and 95% CI by model_type from interpretation_stats_df for each model_type
interpretation_stats_df["model_type"] = interpretation_stats_df["model_name"].map(lambda x: model_metadata[x]["model_type"])

# get only generalist closed  models
generalist_closed_model_stats = interpretation_stats_df[interpretation_stats_df["model_type"] == "generalist closed"]

# get only generalist open models
generalist_open_model_stats = interpretation_stats_df[interpretation_stats_df["model_type"] == "generalist open"]

# get only biomedical open models
biomedical_open_model_stats = interpretation_stats_df[interpretation_stats_df["model_type"] == "biomedical open"]


In [31]:
generalist_closed_model_stats_df = calculate_model_stats(generalist_closed_model_stats, method_name = "generalist closed")

generalist_open_model_stats_df = calculate_model_stats(generalist_open_model_stats, method_name = "generalist open")

biomedical_open_model_stats_df = calculate_model_stats(biomedical_open_model_stats, method_name = "biomedical open")

In [32]:
generalist_closed_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.843810,1.451531,2.236088,benefit_answer,generalist closed
1,0.159048,-0.033614,0.351710,rigor_answer,generalist closed
2,0.702857,0.261401,1.144314,importance_answer,generalist closed
3,1.415238,0.788857,2.041619,full_text_answer,generalist closed
4,1.580952,1.324845,1.837060,another_trial_answer,generalist closed


In [33]:
generalist_open_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,2.269491,1.804069,2.734914,benefit_answer,generalist open
1,0.240833,0.067090,0.414576,rigor_answer,generalist open
2,1.081667,0.785088,1.378245,importance_answer,generalist open
3,1.456667,0.834420,2.078913,full_text_answer,generalist open
4,1.305394,0.744591,1.866198,another_trial_answer,generalist open


In [34]:
biomedical_open_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.820957,0.619193,3.022722,benefit_answer,biomedical open
1,-0.013128,-0.124579,0.098324,rigor_answer,biomedical open
2,0.518704,-0.616388,1.653796,importance_answer,biomedical open
3,1.556339,0.378153,2.734525,full_text_answer,biomedical open
4,1.156642,0.518521,1.794764,another_trial_answer,biomedical open


In [35]:
average_by_model_type = pd.concat([generalist_open_model_stats_df, biomedical_open_model_stats_df, generalist_closed_model_stats_df], ignore_index=True)

average_by_model_type

,mean_diff,ci_lower,ci_upper,metric,method
0,2.269491,1.804069,2.734914,benefit_answer,generalist open
1,0.240833,0.067090,0.414576,rigor_answer,generalist open
2,1.081667,0.785088,1.378245,importance_answer,generalist open
3,1.456667,0.834420,2.078913,full_text_answer,generalist open
4,1.305394,0.744591,1.866198,another_trial_answer,generalist open
5,1.820957,0.619193,3.022722,benefit_answer,biomedical open
6,-0.013128,-0.124579,0.098324,rigor_answer,biomedical open
7,0.518704,-0.616388,1.653796,importance_answer,biomedical open
8,1.556339,0.378153,2.734525,full_text_answer,biomedical open
9,1.156642,0.518521,1.794764,another_trial_answer,biomedical open


In [36]:
#combine all the dataframes
model_stats_final_df = pd.concat([human_expert_stats_df, model_stats_df, average_by_model_type], ignore_index=True)
#drop "_answer" from the values in metric column
model_stats_final_df['metric'] = model_stats_final_df['metric'].str.replace('_answer', '')

model_stats_final_df

,mean_diff,ci_lower,ci_upper,metric,method
0,0.710000,0.070000,1.350000,benefit,human experts
1,-0.590000,-1.130000,-0.050000,rigor,human experts
2,-0.380000,-0.950000,0.190000,importance,human experts
3,0.770000,0.080000,1.470000,full_text,human experts
4,0.640000,-0.030000,1.310000,another_trial,human experts
5,1.991332,1.567488,2.415176,benefit,all LLMs
6,0.134005,0.032919,0.235091,rigor,all LLMs
7,0.782012,0.387193,1.176831,importance,all LLMs
8,1.471335,1.026661,1.916010,full_text,all LLMs
9,1.345742,1.051612,1.639871,another_trial,all LLMs


### Plots

In [37]:
# Create a mapping for custom facet titles
facet_title_mapping = {
    'benefit': 'Treatment Benefit',
    'rigor': 'Study Rigor',
    'importance': 'Study Importance',
    'full_text': 'Interest to Read Full-Text',
    'another_trial': 'Interest to Run Another Trial'
}

# Define the desired order for the facets
facet_order = ['Treatment Benefit', 'Study Rigor', 'Study Importance', 'Interest to Read Full-Text', 'Interest to Run Another Trial']

color_mapping = {
    'human experts': '#0868ac', 
    'all LLMs': '#43a2ca',  
    'generalist closed': '#7bccc4',  
    'generalist open': '#bae4bc',  
    'biomedical open': '#E3F4D4'
}

method_order = ['human experts', 'all LLMs', 'generalist closed', 'generalist open', 'biomedical open']

# Apply the mapping as a calculated field
chart_data = model_stats_final_df.copy()
chart_data['metric'] = chart_data['metric'].map(facet_title_mapping)

# Configure global font sizes
chart_config = {
    "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
    "header": {"labelFontSize": 20, "titleFontSize": 22},  # Facet headers
    "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
    "text": {"fontSize": 20},  # Text mark size
}

# Bar chart
bars = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('method:N', title=None, axis=alt.Axis(labelAngle=-45), sort=method_order),
    y=alt.Y('mean_diff:Q', title='Mean Difference'),
    color=alt.Color('method:N', title='Method', legend=None, scale=alt.Scale(domain=list(color_mapping.keys()), range=list(color_mapping.values())))
).properties(
    width=300,  # Set the width to 300 pixels
    height=300  # Set the height to 300 pixels
)

# Error bars
error_bars = alt.Chart(chart_data).mark_errorbar().encode(
    alt.X("method:N", sort=method_order),
    alt.Y("ci_lower:Q").title("Mean Difference"),
    alt.Y2("ci_upper:Q"),
    strokeWidth=alt.value(2),
    color=alt.value('gray')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    fontWeight='bold',
    dy=alt.expr(expr=alt.expr.if_(alt.datum.mean_diff >= 0, -1, 20))  # Adjust the position of the text    
).encode(
    text=alt.Text('mean_diff:Q', format='.2f'),
    color=alt.value('black')  # Set text color to black
)

# Combine layers and facet
chart = alt.layer(bars, error_bars, text, data=chart_data).facet(
    column=alt.Column('metric:N', title=None, sort=facet_order),
).configure(**chart_config)  # Apply the global configuration

# save to html
chart.save("./plots/interpretation_by_measures.html")

chart

alt.FacetChart(...)

In [38]:
# same plot as above but for only top-6 spin detection accuracy models compare in regards to interpretation

claude_sonnet_diff_data = "eval_outputs/claude_3.5-sonnet/claude_3.5-sonnet_interpretation_outputs.json"
gpt4o_mini_diff_data = "eval_outputs/gpt4o-mini/gpt4o-mini_interpretation_outputs.json"
gemini_flash_8b_diff_data = "eval_outputs/gemini_1.5_flash-8B/gemini_1.5_flash-8B_interpretation_outputs.json"
llama3_8b_diff_data = "eval_outputs/llama3_instruct-8B/llama3_instruct-8B_interpretation_outputs.json"
llama3_70b_diff_data = "eval_outputs/llama3_instruct-70B/llama3_instruct-70B_interpretation_outputs.json"
openbiollm_70b_diff_data = "eval_outputs/openbiollm-70B/openbiollm-70B_interpretation_outputs.json"

# read all the data
claude_sonnet_data = pd.read_json(claude_sonnet_diff_data, orient="record")
gpt4o_mini_data = pd.read_json(gpt4o_mini_diff_data, orient="record")
gemini_flash_8b_data = pd.read_json(gemini_flash_8b_diff_data, orient="record")
llama3_8b_data = pd.read_json(llama3_8b_diff_data, orient="record")
llama3_70b_data = pd.read_json(llama3_70b_diff_data, orient="record")
openbiollm_70b_data = pd.read_json(openbiollm_70b_diff_data, orient="record")

def calculate_ci(data):
    mean_diff = np.mean(data)  # Calculate the mean
    std_dev = np.std(data, ddof=1)  # Use ddof=1 for sample standard deviation  # Calculate the standard deviation
    n = len(data)  # Sample size

    # Calculate the margin of error for 95% CI (z = 1.96)
    z = 1.96
    margin_of_error = z * (std_dev / sqrt(n))

    # Calculate the 95% Confidence Interval
    ci_lower = mean_diff - margin_of_error
    ci_upper = mean_diff + margin_of_error

    return ci_lower, ci_upper

def get_metrics(df):
    column_names = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer"]
    
    diff_metrics = []
    
    for col in column_names:
        df_copy = df.copy()
        
        # Remove rows where the column has 'Error' or empty string
        mask = df_copy[col].apply(lambda x: isinstance(x, str) and ("Error" in x or x == ""))
        if mask.any():
            error_pmids = df_copy.loc[mask, 'PMID'].unique().tolist()
            print(f"Column '{col}' has some 'Error' or empty string values. Removing these rows...")
            print(f"PMIDs with errors: {error_pmids}")
            df_copy = df_copy[~df_copy['PMID'].isin(error_pmids)]
            print(f"Remaining rows after cleanup: {len(df_copy)}")
        
        # Ensure the column is converted to numeric, coercing errors
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

        # For each group by PMID, get the column values difference (abstract_type == spin minus abstract_type == non-spin)
        grouped = df_copy.pivot_table(index='PMID', columns='abstract_type', values=col, aggfunc='mean')
        diff = grouped['spin'] - grouped['no_spin']
        diffs = diff.tolist()

        # calculate confidence interval
        ci_lower, ci_upper = calculate_ci(diffs)

        diff_metrics.append({
            "metric": col,
            "mean_diff": diff.mean(),
            "ci_lower": ci_lower,
            "ci_upper": ci_upper
        })

    return diff_metrics


# get metrics for each model
# save to dataframe for each model
model_data = {
    "Claude 3.5 Sonnet": claude_sonnet_data,
    "GPT4o Mini": gpt4o_mini_data,
    "Gemini1.5 Flash 8B": gemini_flash_8b_data,
    "Llama3 8B": llama3_8b_data,
    "Llama3 70B": llama3_70b_data,
    "OpenBioLLM 70B": openbiollm_70b_data
}
df_for_chart = pd.DataFrame()
for model_name, df in model_data.items():
    diff_metrics = get_metrics(df)
    # create a dataframe
    df_model = pd.DataFrame(diff_metrics)
    df_model["method"] = model_name
    df_for_chart = pd.concat([df_for_chart, df_model], ignore_index=True)

df_for_chart = pd.concat([df_for_chart, human_expert_stats_df], ignore_index=True)
#drop "_answer" from the values in metric column
df_for_chart['metric'] = df_for_chart['metric'].str.replace('_answer', '')

df_for_chart

Column 'full_text_answer' has some 'Error' or empty string values. Removing these rows...
PMIDs with errors: [11261827, 12177098, 15947110, 16148021, 16314619, 17179098, 18794551, 20087643, 20530276, 21471562, 22112969, 9093724]
Remaining rows after cleanup: 36


,metric,mean_diff,ci_lower,ci_upper,method
0,benefit,2.500000,1.978884,3.021116,Claude 3.5 Sonnet
1,rigor,-0.166667,-0.302308,-0.031026,Claude 3.5 Sonnet
2,importance,-0.633333,-0.872616,-0.394051,Claude 3.5 Sonnet
3,full_text,3.233333,2.679892,3.786775,Claude 3.5 Sonnet
4,another_trial,2.866667,2.210635,3.522698,Claude 3.5 Sonnet
5,benefit,3.566667,2.989786,4.143547,GPT4o Mini
6,rigor,1.466667,1.118410,1.814923,GPT4o Mini
7,importance,2.733333,2.283300,3.183367,GPT4o Mini
8,full_text,3.933333,3.402286,4.464381,GPT4o Mini
9,another_trial,3.866667,3.281804,4.451529,GPT4o Mini


In [39]:

# Create a mapping for custom facet titles
facet_title_mapping = {
    'benefit': 'Treatment Benefit',
    'rigor': 'Study Rigor',
    'importance': 'Study Importance',
    'full_text': 'Interest to Read Full-Text',
    'another_trial': 'Interest to Run Another Trial'
}

# Define the desired order for the facets
facet_order = ['Treatment Benefit', 'Study Rigor', 'Study Importance', 'Interest to Read Full-Text', 'Interest to Run Another Trial']

color_mapping = {
    'human experts': '#0868ac', 
    'Claude 3.5 Sonnet': '#43a2ca',  
    'GPT4o Mini': '#7bccc4',  
    'Gemini1.5 Flash 8B': '#bae4bc',  
    'Llama3 8B': '#E3F4D4',  
    'Llama3 70B': '#fdd49e',  
    'OpenBioLLM 70B': '#fdae61'
}

method_order = ['human experts', 'Claude 3.5 Sonnet', 'GPT-4o Mini', 'Gemini1.5 Flash 8B', 'Llama3 8B', 'Llama3 70B', 'OpenBioLLM 70B']

# Apply the mapping as a calculated field
chart_data = df_for_chart.copy()
chart_data['metric'] = chart_data['metric'].map(facet_title_mapping)

# Configure global font sizes
chart_config = {
    "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
    "header": {"labelFontSize": 20, "titleFontSize": 22},  # Facet headers
    "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
    "text": {"fontSize": 20},  # Text mark size
}

# Bar chart
bars = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('method:N', title=None, axis=alt.Axis(labelAngle=-45), sort=method_order),
    y=alt.Y('mean_diff:Q', title='Mean Difference'),
    color=alt.Color('method:N', title='Method', legend=None, scale=alt.Scale(domain=list(color_mapping.keys()), range=list(color_mapping.values())))
).properties(
    width=300,  # Set the width to 300 pixels
    height=300  # Set the height to 300 pixels
)

# Error bars
error_bars = alt.Chart(chart_data).mark_errorbar().encode(
    alt.X("method:N", sort=method_order),
    alt.Y("ci_lower:Q").title("Mean Difference"),
    alt.Y2("ci_upper:Q"),
    strokeWidth=alt.value(2),
    color=alt.value('gray')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    fontWeight='bold',
    dy=alt.expr(expr=alt.expr.if_(alt.datum.mean_diff >= 0, -1, 22))  # Adjust the position of the text    
).encode(
    text=alt.Text('mean_diff:Q', format='.2f'),
    color=alt.value('black')  # Set text color to black
)

# Combine layers and facet
chart = alt.layer(bars, error_bars, text, data=chart_data).facet(
    column=alt.Column('metric:N', title=None, sort=facet_order),
).configure(**chart_config)  # Apply the global configuration

# save to html
chart.save("./plots/top_6_models_interpretation_by_measures.html")

chart

alt.FacetChart(...)

### Mitigation Strategies Results

In [40]:
model_stats_df['method'] = 'baseline'

In [41]:
# gold_labelled_interpretation_overall_metrics.json was manually created
gold_labelled_interpretation_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/gold_labelled_interpretation_overall_metrics_all_new.json", orient="index")

gold_labelled_interpretation_stats_df["model_name"] = gold_labelled_interpretation_stats_df.index
gold_labelled_interpretation_stats_df["model_type"] = gold_labelled_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_type"])
gold_labelled_interpretation_stats_df["model_size_in_b"] = gold_labelled_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_size_in_b"])
# remove index
gold_labelled_interpretation_stats_df.reset_index(drop=True, inplace=True)

print(f"Number of models: {len(gold_labelled_interpretation_stats_df)}")

gold_labelled_interpretation_stats_df.sort_index(inplace=True) # alphabetical order

Number of models: 22


In [42]:
# calculate the average of all model metrics and calculate 95% CI
gold_labelled_model_stats_df = calculate_model_stats(gold_labelled_interpretation_stats_df, method_name = "+ ref labels")
gold_labelled_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.046383,0.677622,1.415144,benefit_answer,+ ref labels
1,-1.066753,-1.419150,-0.714357,rigor_answer,+ ref labels
2,-0.447535,-0.731092,-0.163978,importance_answer,+ ref labels
3,-0.192852,-0.670337,0.284633,full_text_answer,+ ref labels
4,0.341208,0.019668,0.662749,another_trial_answer,+ ref labels


In [43]:
# model_output_labelled_interpretation_overall_metrics.json was manually created
model_output_labelled_interpretation_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/model_output_labelled_interpretation_overall_metrics_all_new.json", orient="index")

model_output_labelled_interpretation_stats_df["model_name"] = model_output_labelled_interpretation_stats_df.index
model_output_labelled_interpretation_stats_df["model_type"] = model_output_labelled_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_type"])
model_output_labelled_interpretation_stats_df["model_size_in_b"] = model_output_labelled_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_size_in_b"])
# remove index
model_output_labelled_interpretation_stats_df.reset_index(drop=True, inplace=True)

print(f"Number of models: {len(model_output_labelled_interpretation_stats_df)}")

model_output_labelled_interpretation_stats_df.sort_index(inplace=True) # alphabetical order

Number of models: 22


In [44]:
# calculate the average of all model metrics and calculate 95% CI
model_output_labelled_model_stats_df = calculate_model_stats(model_output_labelled_interpretation_stats_df, method_name = "+ model labels")
model_output_labelled_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,2.000798,1.472238,2.529357,benefit_answer,+ model labels
1,0.015168,-0.122318,0.152655,rigor_answer,+ model labels
2,0.548827,0.320766,0.776888,importance_answer,+ model labels
3,0.896187,0.611265,1.181110,full_text_answer,+ model labels
4,1.189832,0.825996,1.553667,another_trial_answer,+ model labels


In [45]:
# combined_detection_interpretation_overall_metrics.json was manually created
combined_interpretation_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/combined_detection_interpretation_overall_metrics_all_new.json", orient="index")

combined_interpretation_stats_df["model_name"] = combined_interpretation_stats_df.index
combined_interpretation_stats_df["model_type"] = combined_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_type"])
combined_interpretation_stats_df["model_size_in_b"] = combined_interpretation_stats_df.index.map(lambda x: model_metadata[x]["model_size_in_b"])
# remove index
combined_interpretation_stats_df.reset_index(drop=True, inplace=True)

print(f"Number of models: {len(combined_interpretation_stats_df)}")

combined_interpretation_stats_df.sort_index(inplace=True) # alphabetical order
# combined_interpretation_stats_df

Number of models: 22


In [46]:
# calculate the average of all model metrics and calculate 95% CI
combined_interpretation_stats_df = calculate_model_stats(combined_interpretation_stats_df, method_name = "detect + interpret")
combined_interpretation_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.095852,0.822521,1.369182,benefit_answer,detect + interpret
1,-0.241909,-0.491878,0.008060,rigor_answer,detect + interpret
2,0.377618,-0.065068,0.820304,importance_answer,detect + interpret
3,0.285100,0.013420,0.556779,full_text_answer,detect + interpret
4,0.773424,0.525767,1.021080,another_trial_answer,detect + interpret


In [47]:
all_results = pd.concat([human_expert_stats_df, model_stats_df, gold_labelled_model_stats_df, model_output_labelled_model_stats_df, combined_interpretation_stats_df], ignore_index=True)

all_results

,mean_diff,ci_lower,ci_upper,metric,method
0,0.710000,0.070000,1.350000,benefit_answer,human experts
1,-0.590000,-1.130000,-0.050000,rigor_answer,human experts
2,-0.380000,-0.950000,0.190000,importance_answer,human experts
3,0.770000,0.080000,1.470000,full_text_answer,human experts
4,0.640000,-0.030000,1.310000,another_trial_answer,human experts
5,1.991332,1.567488,2.415176,benefit_answer,baseline
6,0.134005,0.032919,0.235091,rigor_answer,baseline
7,0.782012,0.387193,1.176831,importance_answer,baseline
8,1.471335,1.026661,1.916010,full_text_answer,baseline
9,1.345742,1.051612,1.639871,another_trial_answer,baseline


In [48]:
# Create a mapping for custom facet titles
facet_title_mapping = {
    'benefit_answer': 'Treatment Benefit',
    'rigor_answer': 'Study Rigor',
    'importance_answer': 'Study Importance',
    'full_text_answer': 'Interest to Read Full-Text',
    'another_trial_answer': 'Interest to Run Another Trial'
}

# Define the desired order for the facets
facet_order = ['Treatment Benefit', 'Study Rigor', 'Study Importance', 'Interest to Read Full-Text', 'Interest to Run Another Trial']

color_mapping = {
    'human experts': '#0868ac',  
    'baseline': '#43a2ca',  
    '+ ref labels': '#7bccc4',  
    '+ model labels': '#bae4bc', 
    'detect + interpret': '#E3F4D4'  
}

method_order = ['human experts', 'baseline', '+ ref labels', '+ model labels', 'detect + interpret']

# Apply the mapping as a calculated field
chart_data = all_results.copy()
chart_data['metric'] = chart_data['metric'].map(facet_title_mapping)

# Configure global font sizes
chart_config = {
    "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
    "header": {"labelFontSize": 20, "titleFontSize": 22},  # Facet headers
    "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
    "text": {"fontSize": 20},  # Text mark size
}

# Bar chart
bars = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('method:N', title=None, axis=alt.Axis(labelAngle=-45), sort = method_order),
    y=alt.Y('mean_diff:Q', title='Mean Difference'),
    color=alt.Color('method:N', title='Method', legend=None, scale=alt.Scale(domain=list(color_mapping.keys()), range=list(color_mapping.values())))
).properties(
    width=300,  # Set the width to 300 pixels
    height=300  # Set the height to 300 pixels
)

# Error bars
error_bars = alt.Chart(chart_data).mark_errorbar().encode(
    alt.X("method:N", sort = method_order),
    alt.Y("ci_lower:Q").title("Mean Difference"),
    alt.Y2("ci_upper:Q"),
    strokeWidth=alt.value(2),
    color=alt.value('gray')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    fontWeight='bold',
    dy=alt.expr(expr=alt.expr.if_(alt.datum.mean_diff >= 0, -1, 22))  # Adjust the position of the text    
).encode(
    text=alt.Text('mean_diff:Q', format='.2f'),
    color=alt.value('black')  # Set text color to black
)

# Combine layers and facet
chart = alt.layer(bars, error_bars, text, data=chart_data).facet(
    column=alt.Column('metric:N', title=None, sort=facet_order),
).configure(**chart_config)  # Apply the global configuration

# save to html
chart.save("./plots/interpretation_by_measures_all_methods.html")

chart

alt.FacetChart(...)

In [49]:
interpretation_stats_df["method_category"] = "baseline"
gold_labelled_interpretation_stats_df["method_category"] = "gold_labelled"
model_output_labelled_interpretation_stats_df["method_category"] = "model_output_labelled"

all_interpretation_stats_df = pd.concat([interpretation_stats_df, gold_labelled_interpretation_stats_df, model_output_labelled_interpretation_stats_df], ignore_index=True)

all_interpretation_stats_df

,benefit_answer_mean_diff,rigor_answer_mean_diff,importance_answer_mean_diff,full_text_answer_mean_diff,another_trial_answer_mean_diff,overall_mean_diff_avg,model_name,model_type,model_size_in_b,method_category
0,3.377778,0.420000,1.100000,2.160000,1.573333,1.726222,olmo2_instruct-7B,generalist open,7.0,baseline
1,2.946667,0.766667,1.753333,3.013333,2.440000,2.184000,olmo2_instruct-13B,generalist open,13.0,baseline
2,1.766667,0.046667,1.620000,1.580000,1.253333,1.253333,mistral_instruct7B,generalist open,7.0,baseline
3,2.226667,0.240000,0.933333,1.120000,1.653333,1.234667,med42-8B,biomedical open,8.0,baseline
4,0.626667,0.026667,0.786667,0.547619,0.496667,0.496857,biomistral7B,biomedical open,7.0,baseline
...,...,...,...,...,...,...,...,...,...,...
61,1.753333,-0.006667,0.533333,1.313333,1.460000,1.010667,gpt4o-mini,generalist closed,NaN,model_output_labelled
62,0.720000,-0.400000,0.393333,0.613333,0.900000,0.445333,gemini_1.5_flash,generalist closed,NaN,model_output_labelled
63,1.353333,-0.720000,0.100000,0.860000,1.113333,0.541333,gemini_1.5_flash-8B,generalist closed,8.0,model_output_labelled
64,1.440000,-0.166667,0.100000,1.613333,1.260000,0.849333,claude_3.5-sonnet,generalist closed,NaN,model_output_labelled


In [50]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# fit the model
model = smf.ols(formula="benefit_answer_mean_diff ~ method_category", 
                            data=all_interpretation_stats_df)
results = model.fit()

print(results.summary())


tukey_oneway = pairwise_tukeyhsd(endog = all_interpretation_stats_df["benefit_answer_mean_diff"], groups = all_interpretation_stats_df["method_category"])

# Display the results
tukey_oneway.summary()

                               OLS Regression Results                               
Dep. Variable:     benefit_answer_mean_diff   R-squared:                       0.151
Model:                                  OLS   Adj. R-squared:                  0.123
Method:                       Least Squares   F-statistic:                     5.404
Date:                      Mon, 15 Sep 2025   Prob (F-statistic):            0.00691
Time:                              10:13:46   Log-Likelihood:                -93.679
No. Observations:                        64   AIC:                             193.4
Df Residuals:                            61   BIC:                             199.8
Df Model:                                 2                                         
Covariance Type:                  nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

group1,group2,meandiff,p-adj,lower,upper,reject
baseline,gold_labelled,nan,nan,nan,nan,False
baseline,model_output_labelled,0.0095,nan,nan,nan,False
gold_labelled,model_output_labelled,nan,nan,nan,nan,False


In [51]:
# Create forest plot that shows the difference in scores between baseline and detect + interpret method for treatment benefit
custom_labels = {
    "alpacare-7B": "AlpaCare 7B",
    "biomedgpt7B": "BioMedGPT 7B",
    "biomistral7B": "BioMistral 7B",
    "claude_3.5-haiku": "Claude3.5 Haiku",
    "claude_3.5-sonnet": "Claude3.5 Sonnet", 
    "gemini_1.5_flash": "Gemini1.5 Flash",
    "gemini_1.5_flash-8B": "Gemini1.5 Flash 8B",
    "gpt4o": "GPT-4o",
    "gpt4o-mini": "GPT-4o Mini",
    "gpt35": "GPT-3.5",
    "llama2_chat-7B": "Llama2 Chat 7B",
    "llama2_chat-13B": "Llama2 Chat 13B",
    "llama2_chat-70B": "Llama2 Chat 70B",
    "llama3_instruct-8B": "Llama3 Instruct 8B",
    "llama3_instruct-70B": "Llama3 Instruct 70B",
    "med42-8B": "Med42 8B",
    "med42-70B": "Med42 70B",
    "mistral_instruct7B": "Mistral Instruct 7B",
    "olmo2_instruct-7B": "Olmo2 Instruct 7B",
    "olmo2_instruct-13B": "Olmo2 Instruct 13B",
    "openbiollm-8B": "OpenBioLLM 8B",
    "openbiollm-70B": "OpenBioLLM 70B"
}

baseline_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/interpretation_overall_metrics_all_new.json", orient="index")
baseline_stats_df["model_name"] = baseline_stats_df.index
baseline_stats_df.drop(["overall_mean_diff_avg", "rigor_answer_mean_diff", "importance_answer_mean_diff", "full_text_answer_mean_diff", "another_trial_answer_mean_diff"], axis=1, inplace=True)
# remove index
baseline_stats_df.reset_index(drop=True, inplace=True)
# rename column name benefit_answer_mean_diff to baseline_benefit
baseline_stats_df.rename(columns={"benefit_answer_mean_diff": "baseline_benefit"}, inplace=True)

combined_interpretation_stats_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/combined_detection_interpretation_overall_metrics_all_new.json", orient="index")
combined_interpretation_stats_df["model_name"] = combined_interpretation_stats_df.index
combined_interpretation_stats_df.drop(["overall_avg", "rigor_answer_mean_diff", "importance_answer_mean_diff", "full_text_answer_mean_diff", "another_trial_answer_mean_diff"], axis=1, inplace=True)
# remove index
combined_interpretation_stats_df.reset_index(drop=True, inplace=True)
# rename column name benefit_answer_mean_diff to detect_interpret_benefit
combined_interpretation_stats_df.rename(columns={"benefit_answer_mean_diff": "detect_interpret_benefit"}, inplace=True)

# merge the dataframes by model_name
combined_methods_stats_df = pd.merge(baseline_stats_df, combined_interpretation_stats_df, on="model_name")
combined_methods_stats_df["model_name_custom"] = combined_methods_stats_df["model_name"].map(custom_labels)

combined_methods_stats_df


,baseline_benefit,model_name,detect_interpret_benefit,model_name_custom
0,3.377778,olmo2_instruct-7B,0.913333,Olmo2 Instruct 7B
1,2.946667,olmo2_instruct-13B,0.640000,Olmo2 Instruct 13B
2,1.766667,mistral_instruct7B,1.193333,Mistral Instruct 7B
3,2.226667,med42-8B,0.920000,Med42 8B
4,0.626667,biomistral7B,1.102564,BioMistral 7B
5,3.401575,openbiollm-8B,0.300000,OpenBioLLM 8B
6,1.663194,llama2_chat-7B,1.200000,Llama2 Chat 7B
7,2.402027,llama2_chat-13B,0.260000,Llama2 Chat 13B
8,1.686667,llama3_instruct-8B,0.626667,Llama3 Instruct 8B
9,1.672932,llama2_chat-70B,0.880000,Llama2 Chat 70B


In [52]:
model_order = ['Claude3.5 Sonnet', 
    'GPT-4o Mini', 'Gemini1.5 Flash 8B', 'Llama3 Instruct 8B', 'Llama3 Instruct 70B', 'OpenBioLLM 70B', 'Med42 70B',
    'GPT-4o', 'Gemini1.5 Flash', 'Olmo2 Instruct 7B', 'AlpaCare 7B', 'Llama2 Chat 70B', 'Med42 8B', 'Claude3.5 Haiku', 'Llama2 Chat 13B',
    'GPT-3.5', 'BioMistral 7B', 'Olmo2 Instruct 13B', 'OpenBioLLM 8B', 'Mistral Instruct 7B', 'Llama2 Chat 7B', 'BioMedGPT 7B']

bar = alt.Chart(combined_methods_stats_df).mark_bar(cornerRadius=10, height=10).encode(
    x=alt.X('detect_interpret_benefit:Q').scale(domain=[-2, 6.5]).title('Mean Difference for Treatment Benefit'),
    x2='baseline_benefit:Q',
    y=alt.Y('model_name_custom:N', title="LLM Name", sort=model_order)  # Specify custom order
)

text_min = alt.Chart(combined_methods_stats_df).mark_text(align='right', dx=-5, fontSize=14, fontWeight='bold').encode(
    x='detect_interpret_benefit:Q',
    y=alt.Y('model_name_custom:N', sort=model_order),
    text=alt.Text('detect_interpret_benefit:Q', format='.2f'),
    color=alt.value('#FF8100')  # Set text color to orange
)

text_max = alt.Chart(combined_methods_stats_df).mark_text(align='left', dx=5, fontSize=14, fontWeight='bold').encode(
    x='baseline_benefit:Q',
    y=alt.Y('model_name_custom:N', sort=model_order),
    text=alt.Text('baseline_benefit:Q', format='.2f')
)

chart = (bar + text_min + text_max).properties(
    width=800,
    config={
        'axis': {
            'labelFontSize': 16,
            'titleFontSize': 18
        }
    })

# # Save to HTML
chart.save("./plots/baseline_detect_interpret_treatment_benefit_diff.html")

chart

alt.LayerChart(...)

## Relationship between spin / spin detection and spin interpretation

Linear Regression with statsmodels Python package

In [53]:
model_metadata = {
    "alpacare-7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "biomedgpt7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "biomistral7B": {"model_type": "biomedical open", "model_size_in_b": 7},
    "claude_3.5-haiku": {"model_type": "generalist closed", "model_size_in_b": None},
    "claude_3.5-sonnet": {"model_type": "generalist closed", "model_size_in_b": None},
    "gemini_1.5_flash": {"model_type": "generalist closed", "model_size_in_b": None},
    "gemini_1.5_flash-8B": {"model_type": "generalist closed", "model_size_in_b": 8},
    "gpt4o": {"model_type": "generalist closed", "model_size_in_b": None},
    "gpt4o-mini": {"model_type": "generalist closed", "model_size_in_b": None},
    "gpt35": {"model_type": "generalist closed", "model_size_in_b": 175},
    "llama2_chat-7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "llama2_chat-13B": {"model_type": "generalist open", "model_size_in_b": 13},
    "llama2_chat-70B": {"model_type": "generalist open", "model_size_in_b": 70},
    "llama3_instruct-8B": {"model_type": "generalist open", "model_size_in_b": 8},
    "llama3_instruct-70B": {"model_type": "generalist open", "model_size_in_b": 70},
    "med42-8B": {"model_type": "biomedical open", "model_size_in_b": 8},
    "med42-70B": {"model_type": "biomedical open", "model_size_in_b": 70},
    "mistral_instruct7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "olmo2_instruct-7B": {"model_type": "generalist open", "model_size_in_b": 7},
    "olmo2_instruct-13B": {"model_type": "generalist open", "model_size_in_b": 13},
    "openbiollm-8B": {"model_type": "biomedical open", "model_size_in_b": 8},
    "openbiollm-70B": {"model_type": "biomedical open", "model_size_in_b": 70}
}

In [54]:
# get all model names
model_names = model_metadata.keys()
# remove alpacare-13B
model_names = [x for x in model_names if x != "alpacare-13B"]

len(model_names)

22

In [55]:
measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer"]
gpt_models = ["gpt4o", "gpt4o-mini", "gpt35"]
huggingface_models = ["alpacare-7B", "biomedgpt7B", "biomistral7B", 
                      "llama2_chat-7B", "llama2_chat-13B", "llama2_chat-70B",
                      "llama3_instruct-8B", "llama3_instruct-70B",
                      "med42-8B", "med42-70B", "mistral_instruct7B", 
                      "olmo2_instruct-7B", "olmo2_instruct-13B",
                      "openbiollm-8B", "openbiollm-70B"]
no_probability_models = ["claude_3.5-haiku", "claude_3.5-sonnet", "gemini_1.5_flash-8B"]
# "gemini_1.5_flash",

def get_is_detection_correct(row):
    if row['abstract_type'] == "spin":
        return row['model_answer'] == "yes"
    else:
        return row['model_answer'] == "no"
    
def get_is_abstract_type_spin(row):
    return row['abstract_type'] == "spin"
    
def detection_probability_gpt(row):
    # find the first instance of "yes" or "no"
    token_probabilties = row['model_log_probabilities']
    for token_prob in token_probabilties:
        if token_prob['token'].lower() == "yes":
            return np.exp(token_prob['logprob'])
        elif token_prob['token'].lower() == "no":
            return np.exp(token_prob['logprob'])
    return None # this should not happen but just in case

def detection_probability_huggingface(row):
    # find the first instance of "yes" or "no"
    token_probabilties = row['model_log_probabilities']
    for token_prob in token_probabilties:
        if token_prob['token_string'].lower() == "yes":
            return token_prob['probability']
        elif token_prob['token_string'].lower() == "no":
            return token_prob['probability']
    return None # this should not happen but just in case


def prepare_data_for_regression(model_names):
    for model_name in tqdm(model_names):
        # print(f"Processing {model_name}...")
        final_data = []
        # detection_output_file_path = f"../data/biomedgpt7B_detection_outputs.json"
        # interpretation_output_file_path = f"../data/biomistral7B_detection_outputs.json"
        detection_output_file_path = f"../data/unspun_abstracts/analysis_gpt/{model_name}_detection_outputs.json"
        interpretation_output_file_path = f"../data/unspun_abstracts/analysis_gpt/{model_name}_interpretation_outputs.json"
        model_detection_data = pd.read_json(detection_output_file_path, orient="records")
        model_interpretation_data = pd.read_json(interpretation_output_file_path, orient="records")

        # merge model_detection_data and model_interpretation_data by PMID and abstract_type
        model_data = pd.merge(model_detection_data, model_interpretation_data, on=['PMID', 'abstract_type'])

        # loop through each row in model_data
        for _, row in model_data.iterrows():
            detection_model_prediction = 1 if row['model_answer'] == "yes" else 0
            is_detection_correct = 1 if get_is_detection_correct(row) else 0
            is_spin_in_abstract = 1 if get_is_abstract_type_spin(row) else 0

            if model_name in gpt_models:
                detection_probability = detection_probability_gpt(row)
            elif model_name in huggingface_models:
                detection_probability = detection_probability_huggingface(row)
            else:
                detection_probability = None
            
            for measure in measures:
                final_data.append({
                    "pmid": row['PMID'],
                    "measure": measure,
                    "is_spin_in_abstract": is_spin_in_abstract,
                    "is_detection_correct": is_detection_correct,
                    "detection_model_prediction": detection_model_prediction,
                    "detection_probability": detection_probability,
                    "interpretation_answer": float(row[measure]) if row[measure] != "" else None
                })
            # calculate the average of the differences
            answers = []
            for measure in measures:
                if row[measure] != "":
                    answers.append(float(row[measure]))
            if len(answers) > 0:
                avg_answer= round(np.mean(answers), 6)
            else:
                avg_answer = None
            # add the average difference to the data
            final_data.append({
                "pmid": row['PMID'],
                "measure": "overall",
                "is_spin_in_abstract": is_spin_in_abstract,
                "is_detection_correct": is_detection_correct,
                "detection_model_prediction": detection_model_prediction,
                "detection_probability": detection_probability,
                "interpretation_answer": avg_answer
            })

        # save the final data to a json file
        json_file_path = f"../data/unspun_abstracts/analysis_gpt/{model_name}_combined_data_all_new.csv"
        save_dataset_to_csv(final_data, json_file_path)

In [56]:
prepare_data_for_regression(model_names=model_names)

  0%|          | 0/22 [00:00<?, ?it/s]

100%|██████████| 22/22 [00:07<00:00,  3.13it/s]


#### Simplest Regression

Is spin in abstract and the measures answers

In [57]:
all_stats = {"benefit_answer":{},
             "rigor_answer": {},
             "importance_answer": {},
             "full_text_answer": {},
             "another_trial_answer": {}}

for model_name in model_names:
    output_string = ""
    csv_file_path = f"../data/unspun_abstracts/analysis_gpt/{model_name}_combined_data_all_new.csv"
    data = pd.read_csv(csv_file_path)

    measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer"]
    for measure in measures:
        # get the data for the current measure
        measure_data = data[data['measure'] == measure]
        nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
        # remove rows with NaN values in interpretation_answer
        measure_data = measure_data.dropna(subset=['interpretation_answer'])

        # check if there are less than 2 rows
        if len(measure_data) < 2:
            continue
        
        # fit the model
        model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract", 
                                    data=measure_data)
        results = model.fit()

        stats = {}

        output_string += f"Model: {model_name} - {measure}\n"
        # print number of rows with NaN value(s)
        output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
        output_string += results.summary().as_text()
        output_string += "\n"

        stats["coef"] = results.params[1]
        stats["ci_lower"] = results.conf_int()[0][1]
        stats["ci_upper"] = results.conf_int()[1][1]
        stats["p_value"] = results.f_pvalue
        stats["r_squared"] = results.rsquared
        stats["adjusted_r_squared"] = results.rsquared_adj

        all_stats[measure][model_name] = stats


/tmp/ipykernel_4096095/4137041081.py:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stats["coef"] = results.params[1]
/tmp/ipykernel_4096095/4137041081.py:38: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stats["ci_lower"] = results.conf_int()[0][1]
/tmp/ipykernel_4096095/4137041081.py:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  stats["ci_upper"] = results.conf_int()[1][1]
/tmp/ipykernel_4096095/4137041081.py:37: FutureWarn

In [58]:
all_stats

{'benefit_answer': {'alpacare-7B': {'coef': np.float64(1.4622641509433967),
   'ci_lower': np.float64(-0.1398537490406404),
   'ci_upper': np.float64(3.064382050927434),
   'p_value': np.float64(0.07313730638680997),
   'r_squared': np.float64(0.03448499438433983),
   'adjusted_r_squared': np.float64(0.023990266062430488)},
  'biomedgpt7B': {'coef': np.float64(0.018991596638654795),
   'ci_lower': np.float64(-0.7457285528395464),
   'ci_upper': np.float64(0.783711746116856),
   'p_value': np.float64(0.9610235826852371),
   'r_squared': np.float64(9.888926920664787e-06),
   'adjusted_r_squared': np.float64(-0.004122301614703616)},
  'biomistral7B': {'coef': np.float64(0.6118421052631561),
   'ci_lower': np.float64(0.36763613608203394),
   'ci_upper': np.float64(0.8560480744442782),
   'p_value': np.float64(1.3568617703010543e-06),
   'r_squared': np.float64(0.07449419911630184),
   'adjusted_r_squared': np.float64(0.07142961037165396)},
  'claude_3.5-haiku': {'coef': np.float64(1.666666

In [12]:
import json
with open(f"../data/unspun_abstracts/analysis_gpt/simple_regression_summary_all_new.json", "w") as f:
    json.dump(all_stats, f, indent=4)

In [13]:
measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer"]
for measure in measures:
    with open(f"../data/unspun_abstracts/analysis_gpt/{measure}_simple_regression_summary_all_new.json", "w") as f:
        json.dump(all_stats[measure], f, indent=4)

In [14]:
all_stats = {"benefit_answer":{},
             "rigor_answer": {},
             "importance_answer": {},
             "full_text_answer": {},
             "another_trial_answer": {}}

for model_name in model_names:
    output_string = ""
    csv_file_path = f"../data/unspun_abstracts/analysis_gpt/{model_name}_combined_data_all_new.csv"
    data = pd.read_csv(csv_file_path)

    measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer", "overall"]
    for measure in measures:
        # get the data for the current measure
        measure_data = data[data['measure'] == measure]
        nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
        # remove rows with NaN values in interpretation_answer
        measure_data = measure_data.dropna(subset=['interpretation_answer'])

        # check if there are less than 2 rows
        if len(measure_data) < 2:
            continue
        
        # fit the model
        model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract", 
                                    data=measure_data)
        results = model.fit()

        output_string += f"Model: {model_name} - {measure}\n"
        # print number of rows with NaN value(s)
        output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
        output_string += results.summary().as_text()
        output_string += "\n"

    # save the model summary
    with open(f"../data/unspun_abstracts/analysis_gpt/{model_name}_simple_regression_summary_all_new.txt", "w") as f:
        f.write(output_string)

/home/zhang.yuchen/.conda/envs/jup/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/home/zhang.yuchen/.conda/envs/jup/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1871: RuntimeWarning: invalid value encountered in scalar divide
  return self.mse_model/self.mse_resid
/home/zhang.yuchen/.conda/envs/jup/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
/home/zhang.yuchen/.conda/envs/jup/lib/python3.12/site-packages/statsmodels/stats/stattools.py:50: RuntimeWarning: invalid value encountered in scalar divide
  dw = np.sum(diff_resids**2, axis=axis) / np.sum(resids**2, axis=axis)
/home/zhang.yuchen/.conda/envs/jup/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invali

##### Forest Plot for "Benefit" Linear Regression Results

In [22]:
import altair as alt
import pandas as pd

custom_labels = {
    "alpacare-7B": "AlpaCare 7B",
    "biomedgpt7B": "BioMedGPT 7B",
    "biomistral7B": "BioMistral 7B",
    "claude_3.5-haiku": "Claude3.5 Haiku",
    "claude_3.5-sonnet": "Claude3.5 Sonnet", 
    "gemini_1.5_flash": "Gemini1.5 Flash",
    "gemini_1.5_flash-8B": "Gemini1.5 Flash 8B",
    "gpt4o": "GPT-4o",
    "gpt4o-mini": "GPT-4o Mini",
    "gpt35": "GPT-3.5",
    "llama2_chat-7B": "Llama2 Chat 7B",
    "llama2_chat-13B": "Llama2 Chat 13B",
    "llama2_chat-70B": "Llama2 Chat 70B",
    "llama3_instruct-8B": "Llama3 Instruct 8B",
    "llama3_instruct-70B": "Llama3 Instruct 70B",
    "med42-8B": "Med42 8B",
    "med42-70B": "Med42 70B",
    "mistral_instruct7B": "Mistral Instruct 7B",
    "olmo2_instruct-7B": "Olmo2 Instruct 7B",
    "olmo2_instruct-13B": "Olmo2 Instruct 13B",
    "openbiollm-8B": "OpenBioLLM 8B",
    "openbiollm-70B": "OpenBioLLM 70B"
}

# Load the JSON data into a DataFrame
# this json file was manually created
regression_results_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/benefit_answer_simple_regression_summary_all_new.json", orient="index")

# Ensure index is reset and available as a column
regression_results_df.reset_index(inplace=True)
regression_results_df = regression_results_df.rename(columns={'index': 'model_name'})

regression_results_df["model_name_custom"] = regression_results_df["model_name"].map(custom_labels)

# Create the Altair chart
points = alt.Chart(regression_results_df).mark_point(
    filled=True,
    color='red',
    size=50  # Increase point size
).encode(
    x=alt.X('coef:Q').title('Coefficient'),
    y=alt.Y('model_name_custom:N').title('LLM Name').sort(
        field='coef',  # Sort by coefficient values
        order='descending'
    )
).properties(
    width=600,
    height=300
)

# Add error bars
error_bars = points.mark_rule(
    strokeWidth=2  # Increase width of error bars
).encode(
    x='ci_lower:Q',
    x2='ci_upper:Q',
    size=alt.value(2)  # Set the width of error bars
)

# Add vertical line at x = 0.71
vertical_line = alt.Chart(pd.DataFrame({'x': [0.71]})).mark_rule(
    color='blue',
    strokeDash=[4, 4],  # Make it dashed
    strokeWidth=2
).encode(
    x='x:Q',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Add label for vertical line
label = alt.Chart(pd.DataFrame({'x': [0.71], 'y': [regression_results_df['model_name_custom'].iloc[0]]})).mark_text(
    text='Human Experts',
    align='center',
    dx=5,  # Adjust text position
    dy=-10,
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    y=alt.value(0),  # Adjust position if necessary
    color=alt.value('#0868ac')  # Specify the color directly
)

# Define custom x-axis labels
custom_labels_df = pd.DataFrame({
    'x': [-2.5, 4.2],
    'text': ['Less susceptible to spin', 'More susceptible to spin']
})

# Define custom x-axis labels
left_arrow_df = pd.DataFrame({
    'x': [-4],
    'text': ['←']
})

# Define custom x-axis labels
right_arrow_df = pd.DataFrame({
    'x': [6.5],
    'text': ['→']
})

# Create a text layer for custom x-axis labels
custom_x_labels = alt.Chart(custom_labels_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=24, # adjust horizontal positioning
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_left_arrow = alt.Chart(left_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=10, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_right_arrow = alt.Chart(right_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=0, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Combine all layers, including the new x-axis labels
chart = alt.layer(error_bars, points, vertical_line, label, custom_x_labels, custom_x_left_arrow, custom_x_right_arrow).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=20
)

# # Save to HTML
chart.save("./plots/simple_regression_benefit_data.html")

chart


alt.LayerChart(...)

In [ ]:
# rest of the questions

regression_results_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/another_trial_answer_simple_regression_summary_all_new.json", orient="index")

# Ensure index is reset and available as a column
regression_results_df.reset_index(inplace=True)
regression_results_df = regression_results_df.rename(columns={'index': 'model_name'})

regression_results_df["model_name_custom"] = regression_results_df["model_name"].map(custom_labels)

# Create the Altair chart
points = alt.Chart(regression_results_df).mark_point(
    filled=True,
    color='red',
    size=50  # Increase point size
).encode(
    x=alt.X('coef:Q').title('Coefficient'),
    y=alt.Y('model_name_custom:N').title('LLM Name').sort(
        field='coef',  # Sort by coefficient values
        order='descending'
    )
).properties(
    width=600,
    height=300
)

# Add error bars
error_bars = points.mark_rule(
    strokeWidth=2  # Increase width of error bars
).encode(
    x='ci_lower:Q',
    x2='ci_upper:Q',
    size=alt.value(2)  # Set the width of error bars
)

# Add vertical line
vertical_line = alt.Chart(pd.DataFrame({'x': [0.640000]})).mark_rule(
    color='blue',
    strokeDash=[4, 4],  # Make it dashed
    strokeWidth=2
).encode(
    x='x:Q',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Add label for vertical line
label = alt.Chart(pd.DataFrame({'x': [0.640000], 'y': [regression_results_df['model_name_custom'].iloc[0]]})).mark_text(
    text='Human Experts',
    align='center',
    dx=5,  # Adjust text position
    dy=-10,
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    y=alt.value(0),  # Adjust position if necessary
    color=alt.value('#0868ac')  # Specify the color directly
)

# Define custom x-axis labels
custom_labels_df = pd.DataFrame({
    'x': [-3.5, 4.7],
    'text': ['Less susceptible to spin', 'More susceptible to spin']
})

# Define custom x-axis labels
left_arrow_df = pd.DataFrame({
    'x': [-5.5],
    'text': ['←']
})

# Define custom x-axis labels
right_arrow_df = pd.DataFrame({
    'x': [7.5],
    'text': ['→']
})

# Create a text layer for custom x-axis labels
custom_x_labels = alt.Chart(custom_labels_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=24, # adjust horizontal positioning
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_left_arrow = alt.Chart(left_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=10, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_right_arrow = alt.Chart(right_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=0, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Combine all layers, including the new x-axis labels
chart = alt.layer(error_bars, points, vertical_line, label, custom_x_labels, custom_x_left_arrow, custom_x_right_arrow).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=20
)

# Save to HTML
chart.save("./plots/simple_regression_another_trial_data.html")

chart


alt.LayerChart(...)

In [ ]:
custom_labels = {
    "alpacare-7B": "AlpaCare 7B",
    "biomedgpt7B": "BioMedGPT 7B",
    "biomistral7B": "BioMistral 7B",
    "claude_3.5-haiku": "Claude3.5 Haiku",
    "claude_3.5-sonnet": "Claude3.5 Sonnet", 
    "gemini_1.5_flash": "Gemini1.5 Flash",
    "gemini_1.5_flash-8B": "Gemini1.5 Flash 8B",
    "gpt4o": "GPT-4o",
    "gpt4o-mini": "GPT-4o Mini",
    "gpt35": "GPT-3.5",
    "llama2_chat-7B": "Llama2 Chat 7B",
    "llama2_chat-13B": "Llama2 Chat 13B",
    "llama2_chat-70B": "Llama2 Chat 70B",
    "llama3_instruct-8B": "Llama3 Instruct 8B",
    "llama3_instruct-70B": "Llama3 Instruct 70B",
    "med42-8B": "Med42 8B",
    "med42-70B": "Med42 70B",
    "mistral_instruct7B": "Mistral Instruct 7B",
    "olmo2_instruct-7B": "Olmo2 Instruct 7B",
    "olmo2_instruct-13B": "Olmo2 Instruct 13B",
    "openbiollm-8B": "OpenBioLLM 8B",
    "openbiollm-70B": "OpenBioLLM 70B"
}

In [ ]:
# rest of the questions

regression_results_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/rigor_answer_simple_regression_summary_all_new.json", orient="index")

# Ensure index is reset and available as a column
regression_results_df.reset_index(inplace=True)
regression_results_df = regression_results_df.rename(columns={'index': 'model_name'})

regression_results_df["model_name_custom"] = regression_results_df["model_name"].map(custom_labels)

# Create the Altair chart
points = alt.Chart(regression_results_df).mark_point(
    filled=True,
    color='red',
    size=50  # Increase point size
).encode(
    x=alt.X('coef:Q').title('Coefficient'),
    y=alt.Y('model_name_custom:N').title('LLM Name').sort(
        field='coef',  # Sort by coefficient values
        order='descending'
    )
).properties(
    width=600,
    height=300
)

# Add error bars
error_bars = points.mark_rule(
    strokeWidth=2  # Increase width of error bars
).encode(
    x='ci_lower:Q',
    x2='ci_upper:Q',
    size=alt.value(2)  # Set the width of error bars
)

# Add vertical line 
vertical_line = alt.Chart(pd.DataFrame({'x': [-0.590000]})).mark_rule(
    color='blue',
    strokeDash=[4, 4],  # Make it dashed
    strokeWidth=2
).encode(
    x='x:Q',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Add label for vertical line
label = alt.Chart(pd.DataFrame({'x': [0.640000], 'y': [regression_results_df['model_name_custom'].iloc[0]]})).mark_text(
    text='Human Experts',
    align='center',
    dx=5,  # Adjust text position
    dy=-10,
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    y=alt.value(0),  # Adjust position if necessary
    color=alt.value('#0868ac')  # Specify the color directly
)

# Define custom x-axis labels
custom_labels_df = pd.DataFrame({
    'x': [-0.66, 0.65],
    'text': ['Less susceptible to spin', 'More susceptible to spin']
})

# Define custom x-axis labels
left_arrow_df = pd.DataFrame({
    'x': [-1],
    'text': ['←']
})

# Define custom x-axis labels
right_arrow_df = pd.DataFrame({
    'x': [1.1],
    'text': ['→']
})

# Create a text layer for custom x-axis labels
custom_x_labels = alt.Chart(custom_labels_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=24, # adjust horizontal positioning
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_left_arrow = alt.Chart(left_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=10, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_right_arrow = alt.Chart(right_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=0, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Combine all layers, including the new x-axis labels
chart = alt.layer(error_bars, points, vertical_line, label, custom_x_labels, custom_x_left_arrow, custom_x_right_arrow).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=20
)

# Save to HTML
chart.save("./plots/simple_regression_rigor_data.html")

chart


alt.LayerChart(...)

In [ ]:
# rest of the questions

regression_results_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/full_text_answer_simple_regression_summary_all_new.json", orient="index")

# Ensure index is reset and available as a column
regression_results_df.reset_index(inplace=True)
regression_results_df = regression_results_df.rename(columns={'index': 'model_name'})

regression_results_df["model_name_custom"] = regression_results_df["model_name"].map(custom_labels)

# Create the Altair chart
points = alt.Chart(regression_results_df).mark_point(
    filled=True,
    color='red',
    size=50  # Increase point size
).encode(
    x=alt.X('coef:Q').title('Coefficient'),
    y=alt.Y('model_name_custom:N').title('LLM Name').sort(
        field='coef',  # Sort by coefficient values
        order='descending'
    )
).properties(
    width=600,
    height=300
)

# Add error bars
error_bars = points.mark_rule(
    strokeWidth=2  # Increase width of error bars
).encode(
    x='ci_lower:Q',
    x2='ci_upper:Q',
    size=alt.value(2)  # Set the width of error bars
)

# Add vertical line
vertical_line = alt.Chart(pd.DataFrame({'x': [0.640000]})).mark_rule(
    color='blue',
    strokeDash=[4, 4],  # Make it dashed
    strokeWidth=2
).encode(
    x='x:Q',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Add label for vertical line
label = alt.Chart(pd.DataFrame({'x': [0.640000], 'y': [regression_results_df['model_name_custom'].iloc[0]]})).mark_text(
    text='Human Experts',
    align='center',
    dx=5,  # Adjust text position
    dy=-10,
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    y=alt.value(0),  # Adjust position if necessary
    color=alt.value('#0868ac')  # Specify the color directly
)

# Define custom x-axis labels
custom_labels_df = pd.DataFrame({
    'x': [-0.8, 4.2],
    'text': ['Less susceptible to spin', 'More susceptible to spin']
})

# Define custom x-axis labels
left_arrow_df = pd.DataFrame({
    'x': [-2],
    'text': ['←']
})

# Define custom x-axis labels
right_arrow_df = pd.DataFrame({
    'x': [5.8],
    'text': ['→']
})

# Create a text layer for custom x-axis labels
custom_x_labels = alt.Chart(custom_labels_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=24, # adjust horizontal positioning
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_left_arrow = alt.Chart(left_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=10, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_right_arrow = alt.Chart(right_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=0, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Combine all layers, including the new x-axis labels
chart = alt.layer(error_bars, points, vertical_line, label, custom_x_labels, custom_x_left_arrow, custom_x_right_arrow).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=20
)

# Save to HTML
chart.save("./plots/simple_regression_full_text_data.html")

chart


alt.LayerChart(...)

In [ ]:
# rest of the questions

regression_results_df = pd.read_json("../data/unspun_abstracts/analysis_gpt/importance_answer_simple_regression_summary_all_new.json", orient="index")

# Ensure index is reset and available as a column
regression_results_df.reset_index(inplace=True)
regression_results_df = regression_results_df.rename(columns={'index': 'model_name'})

regression_results_df["model_name_custom"] = regression_results_df["model_name"].map(custom_labels)

# Create the Altair chart
points = alt.Chart(regression_results_df).mark_point(
    filled=True,
    color='red',
    size=50  # Increase point size
).encode(
    x=alt.X('coef:Q').title('Coefficient'),
    y=alt.Y('model_name_custom:N').title('LLM Name').sort(
        field='coef',  # Sort by coefficient values
        order='descending'
    )
).properties(
    width=600,
    height=300
)

# Add error bars
error_bars = points.mark_rule(
    strokeWidth=2  # Increase width of error bars
).encode(
    x='ci_lower:Q',
    x2='ci_upper:Q',
    size=alt.value(2)  # Set the width of error bars
)

# Add vertical line
vertical_line = alt.Chart(pd.DataFrame({'x': [0.640000]})).mark_rule(
    color='blue',
    strokeDash=[4, 4],  # Make it dashed
    strokeWidth=2
).encode(
    x='x:Q',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Add label for vertical line
label = alt.Chart(pd.DataFrame({'x': [0.640000], 'y': [regression_results_df['model_name_custom'].iloc[0]]})).mark_text(
    text='Human Experts',
    align='center',
    dx=5,  # Adjust text position
    dy=-10,
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    y=alt.value(0),  # Adjust position if necessary
    color=alt.value('#0868ac')  # Specify the color directly
)

# Define custom x-axis labels
custom_labels_df = pd.DataFrame({
    'x': [-2.8, 2.2],
    'text': ['Less susceptible to spin', 'More susceptible to spin']
})

# Define custom x-axis labels
left_arrow_df = pd.DataFrame({
    'x': [-4],
    'text': ['←']
})

# Define custom x-axis labels
right_arrow_df = pd.DataFrame({
    'x': [3.8],
    'text': ['→']
})

# Create a text layer for custom x-axis labels
custom_x_labels = alt.Chart(custom_labels_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=24, # adjust horizontal positioning
    fontSize=14,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_left_arrow = alt.Chart(left_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=10, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Create a text layer for custom x-axis labels
custom_x_right_arrow = alt.Chart(right_arrow_df).mark_text(
    align='center',
    baseline='bottom',
    dy=195,  # Adjust vertical positioning
    dx=0, # adjust horizontal positioning
    fontSize=24,
    fontWeight='bold',
).encode(
    x='x:Q',
    text='text:N',
    color=alt.value('#0868ac')  # Specify the color directly
)

# Combine all layers, including the new x-axis labels
chart = alt.layer(error_bars, points, vertical_line, label, custom_x_labels, custom_x_left_arrow, custom_x_right_arrow).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=20
)

# Save to HTML
chart.save("./plots/simple_regression_importance_data.html")

chart


alt.LayerChart(...)

: 

#### Binary Spin Detection Results Version

In [174]:
# for model_name in model_names:
#     output_string = ""
#     csv_file_path = f"./eval_outputs/{model_name}/{model_name}_combined_data.csv"
#     data = pd.read_csv(csv_file_path)

#     measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer", "overall"]
#     for measure in measures:
#         # get the data for the current measure
#         measure_data = data[data['measure'] == measure]
#         nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
#         # remove rows with NaN values in interpretation_answer
#         measure_data = measure_data.dropna(subset=['interpretation_answer'])

#         # check if there are less than 2 rows
#         if len(measure_data) < 2:
#             continue
        
#         # fit the model
#         model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract * is_detection_correct", 
#                                     data=measure_data)
#         results = model.fit()

#         output_string += f"Model: {model_name} - {measure}\n"
#         # print number of rows with NaN value(s)
#         output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
#         output_string += results.summary().as_text()
#         output_string += "\n"

#     # save the model summary
#     with open(f"./eval_outputs/{model_name}/{model_name}_regression_binary_summary.txt", "w") as f:
#         f.write(output_string)

In [175]:
# # what the model predicts rather than whether it was correct or not
# for model_name in model_names:
#     output_string = ""
#     csv_file_path = f"./eval_outputs/{model_name}/{model_name}_combined_data.csv"
#     data = pd.read_csv(csv_file_path)

#     measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer", "overall"]
#     for measure in measures:
#         # get the data for the current measure
#         measure_data = data[data['measure'] == measure]
#         nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
#         # remove rows with NaN values in interpretation_answer
#         measure_data = measure_data.dropna(subset=['interpretation_answer'])

#         # check if there are less than 2 rows
#         if len(measure_data) < 2:
#             continue
        
#         # fit the model
#         model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract * detection_model_prediction", 
#                                     data=measure_data)
#         results = model.fit()

#         output_string += f"Model: {model_name} - {measure}\n"
#         # print number of rows with NaN value(s)
#         output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
#         output_string += results.summary().as_text()
#         output_string += "\n"

#     # save the model summary
#     with open(f"./eval_outputs/{model_name}/{model_name}_regression_binary_direct_model_prediction_summary.txt", "w") as f:
#         f.write(output_string)

#### Probability Spin Detection Results Version

In [176]:
# model_names = gpt_models + huggingface_models # remove no token probability models

# for model_name in model_names:
#     output_string = ""
#     csv_file_path = f"./eval_outputs/{model_name}/{model_name}_combined_data.csv"
#     data = pd.read_csv(csv_file_path)

#     measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer", "overall"]
#     for measure in measures:
#         # get the data for the current measure
#         measure_data = data[data['measure'] == measure]
#         nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
#         # remove rows with NaN values in interpretation_answer
#         measure_data = measure_data.dropna(subset=['interpretation_answer', 'detection_probability'])
        
#         # if is_detection_no_spin_correct == 1, then detection_probability. Otherwise, 1 - detection_probability
#         measure_data['regression_detection_variable'] = measure_data.apply(lambda x: x['detection_probability'] if x['is_detection_correct'] == 1 else 1 - x['detection_probability'], axis=1)
#         # check if there are less than 2 rows
#         if len(measure_data) < 2:
#             continue

#         # fit the model
#         model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract * regression_detection_variable",
#                                     data=measure_data)
#         results = model.fit()

#         output_string += f"Model: {model_name} - {measure}\n"
#         # print number of rows with NaN value(s)
#         output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
#         output_string += results.summary().as_text()
#         output_string += "\n"

#     # save the model summary
#     with open(f"./eval_outputs/{model_name}/{model_name}_regression_probability_summary.txt", "w") as f:
#         f.write(output_string)

In [177]:
# # what the model predicts rather than whether it was correct or not

# model_names = gpt_models + huggingface_models # remove no token probability models

# for model_name in model_names:
#     output_string = ""
#     csv_file_path = f"./eval_outputs/{model_name}/{model_name}_combined_data.csv"
#     data = pd.read_csv(csv_file_path)

#     measures = ["benefit_answer", "rigor_answer", "importance_answer", "full_text_answer", "another_trial_answer", "overall"]
#     for measure in measures:
#         # get the data for the current measure
#         measure_data = data[data['measure'] == measure]
#         nan_rows_number = measure_data['interpretation_answer'].isnull().sum()
#         # remove rows with NaN values in interpretation_answer
#         measure_data = measure_data.dropna(subset=['interpretation_answer', 'detection_probability'])
        
#         # if is_detection_no_spin_correct == 1, then detection_probability. Otherwise, 1 - detection_probability
#         measure_data['regression_detection_variable'] = measure_data.apply(lambda x: x['detection_probability'] if x['detection_model_prediction'] == 1 else 1 - x['detection_probability'], axis=1)
#         # check if there are less than 2 rows
#         if len(measure_data) < 2:
#             continue

#         # fit the model
#         model = smf.ols(formula="interpretation_answer ~ is_spin_in_abstract * regression_detection_variable",
#                                     data=measure_data)
#         results = model.fit()

#         output_string += f"Model: {model_name} - {measure}\n"
#         # print number of rows with NaN value(s)
#         output_string += f"Number of rows with NaN value(s) in {model_name}: {nan_rows_number}\n"
#         output_string += results.summary().as_text()
#         output_string += "\n"

#     # save the model summary
#     with open(f"./eval_outputs/{model_name}/{model_name}_regression_probability_direct_model_prediction_summary.txt", "w") as f:
#         f.write(output_string)

## PLS LLM Interpretations

### Average Tokens

In [5]:
# get all model names
model_names = model_metadata.keys()
# remove alpacare-13B
model_names = [x for x in model_names if x != "alpacare-13B"]

len(model_names)

22

In [8]:
# get all PLS outputs from all LLMs
enc = tiktoken.get_encoding("o200k_base") # for gpt-4o and gpt-4o mini

model_token_stats = {}
number_of_tokens_total = [] # for all models
for model_name in model_names:
    csv_file_path = f"./pls_outputs/{model_name}/{model_name}_outputs.csv"
    data = pd.read_csv(csv_file_path)
    # calculate the number of tokens for each row in plain_language_summary
    plain_language_summaries = data['plain_language_summary'].tolist()

    number_of_tokens = []
    for summary in plain_language_summaries:
        token_integers = enc.encode(summary)
        number_of_tokens.append(len(token_integers))
        number_of_tokens_total.append(len(token_integers))

    # average number of tokens
    average_number_of_tokens = np.mean(number_of_tokens)
    # SD of tokens
    sd_number_of_tokens = np.std(number_of_tokens)
    model_token_stats[model_name] = {"average_number_of_tokens": average_number_of_tokens, "sd_number_of_tokens": sd_number_of_tokens}

model_token_stats_df = pd.DataFrame(model_token_stats).T
model_token_stats_df["model_name"] = model_token_stats_df.index

model_token_stats_df

,average_number_of_tokens,sd_number_of_tokens,model_name
alpacare-7B,120.416667,56.264936,alpacare-7B
biomedgpt7B,195.000000,59.498459,biomedgpt7B
biomistral7B,140.766667,73.807941,biomistral7B
claude_3.5-haiku,225.033333,15.084171,claude_3.5-haiku
claude_3.5-sonnet,230.516667,19.185491,claude_3.5-sonnet
gemini_1.5_flash,214.216667,30.953778,gemini_1.5_flash
gemini_1.5_flash-8B,207.700000,37.768285,gemini_1.5_flash-8B
gpt4o,207.850000,44.255254,gpt4o
gpt4o-mini,253.783333,38.084595,gpt4o-mini
gpt35,94.483333,18.754992,gpt35


In [9]:
# get the average across all
average_number_of_tokens = np.mean(number_of_tokens_total)
sd_number_of_tokens = np.std(number_of_tokens_total)

average_number_of_tokens, sd_number_of_tokens

(np.float64(208.10378787878787), np.float64(67.01472464306116))

In [10]:
# average across all models
average_number_of_tokens = model_token_stats_df["average_number_of_tokens"].mean()
sd_number_of_tokens = model_token_stats_df["sd_number_of_tokens"].mean()

average_number_of_tokens, sd_number_of_tokens

(np.float64(208.10378787878787), np.float64(37.62843632260712))

### LLM Evaluation Results

In [13]:
# the following files were manually created
claude_evaluator_results = pd.read_json("./pls_outputs/extended_dataset/_interpretation_eval_results/claude_3.5-sonnet/claude_3.5-sonnet_pls_interpretation_overall_metrics.json", orient="index")
gpt4o_mini_evaluator_results = pd.read_json("./pls_outputs/extended_dataset/_interpretation_eval_results/gpt4o-mini/gpt4o-mini_pls_interpretation_overall_metrics.json", orient="index")
openbiollm_70b_evaluator_results = pd.read_json("./pls_outputs/extended_dataset/_interpretation_eval_results/openbiollm-70B/openbiollm-70B_pls_interpretation_overall_metrics.json", orient="index")

In [14]:
def calculate_confidence_interval(df, df_column_name):
    mean_diff = df[df_column_name].mean()  # Calculate the mean
    std_dev = df[df_column_name].std()  # Calculate the standard deviation
    n = len(df[df_column_name])  # Sample size

    # Calculate the margin of error for 95% CI (z = 1.96)
    z = 1.96
    margin_of_error = z * (std_dev / sqrt(n))

    # Calculate the 95% Confidence Interval
    ci_lower = mean_diff - margin_of_error
    ci_upper = mean_diff + margin_of_error

    return ci_lower, ci_upper

In [15]:
pls_gpt4o_mini_model_stats_df = calculate_model_stats(gpt4o_mini_evaluator_results, method_name = "GPT-4o Mini")

pls_gpt4o_mini_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.692121,1.574447,1.809795,benefit_answer,GPT-4o Mini
1,0.655455,0.605632,0.705277,rigor_answer,GPT-4o Mini
2,1.135758,1.066447,1.205068,importance_answer,GPT-4o Mini
3,1.536364,1.433524,1.639203,full_text_answer,GPT-4o Mini
4,1.681515,1.566895,1.796136,another_trial_answer,GPT-4o Mini


In [16]:
pls_claude_model_stats_df = calculate_model_stats(claude_evaluator_results, method_name = "Claude 3.5 Sonnet")

pls_claude_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,0.801802,0.689977,0.913627,benefit_answer,Claude 3.5 Sonnet
1,-0.189411,-0.270355,-0.108467,rigor_answer,Claude 3.5 Sonnet
2,0.226097,0.145219,0.306976,importance_answer,Claude 3.5 Sonnet
3,0.663469,0.519308,0.807630,full_text_answer,Claude 3.5 Sonnet
4,1.104029,1.008814,1.199244,another_trial_answer,Claude 3.5 Sonnet


In [17]:
pls_openbiollm_model_stats_df = calculate_model_stats(openbiollm_70b_evaluator_results, method_name = "OpenBioLLM 70B")

pls_openbiollm_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.699963,1.559266,1.840660,benefit_answer,OpenBioLLM 70B
1,0.101372,0.067478,0.135266,rigor_answer,OpenBioLLM 70B
2,0.800792,0.704386,0.897198,importance_answer,OpenBioLLM 70B
3,0.166671,0.059299,0.274043,full_text_answer,OpenBioLLM 70B
4,1.475031,1.287530,1.662533,another_trial_answer,OpenBioLLM 70B


In [18]:
# combine two dataframes
all_pls_model_stats_df = pd.concat([pls_gpt4o_mini_model_stats_df, pls_claude_model_stats_df, pls_openbiollm_model_stats_df], ignore_index=True)

all_pls_model_stats_df

,mean_diff,ci_lower,ci_upper,metric,method
0,1.692121,1.574447,1.809795,benefit_answer,GPT-4o Mini
1,0.655455,0.605632,0.705277,rigor_answer,GPT-4o Mini
2,1.135758,1.066447,1.205068,importance_answer,GPT-4o Mini
3,1.536364,1.433524,1.639203,full_text_answer,GPT-4o Mini
4,1.681515,1.566895,1.796136,another_trial_answer,GPT-4o Mini
5,0.801802,0.689977,0.913627,benefit_answer,Claude 3.5 Sonnet
6,-0.189411,-0.270355,-0.108467,rigor_answer,Claude 3.5 Sonnet
7,0.226097,0.145219,0.306976,importance_answer,Claude 3.5 Sonnet
8,0.663469,0.519308,0.807630,full_text_answer,Claude 3.5 Sonnet
9,1.104029,1.008814,1.199244,another_trial_answer,Claude 3.5 Sonnet


In [ ]:
# create altair grouped barchart
# grouped by metric and evaluator

# Create a mapping for custom facet titles
facet_title_mapping = {
    'benefit_answer': 'Benefit',
    'rigor_answer': 'Rigor',
    'importance_answer': 'Importance',
    'full_text_answer': 'Full-Text',
    'another_trial_answer': 'Another Trial'
}

# Define the desired order for the facets
facet_order = ['Benefit', 'Rigor', 'Importance', 'Full-Text', 'Another Trial']

color_mapping = {
    'Claude 3.5 Sonnet': '#0868ac',  
    'GPT-4o Mini': '#43a2ca',  
    'OpenBioLLM 70B': '#7bccc4'
}

method_order = ['Claude 3.5 Sonnet', 'GPT-4o Mini', 'OpenBioLLM 70B']

# Apply the mapping as a calculated field
chart_data = all_pls_model_stats_df.copy()
chart_data['metric'] = chart_data['metric'].map(facet_title_mapping)

# Configure global font sizes
chart_config = {
    "axis": {"labelFontSize": 20, "titleFontSize": 22},  # Axis labels and titles
    "header": {"labelFontSize": 20, "titleFontSize": 22},  # Facet headers
    "legend": {"labelFontSize": 18, "titleFontSize": 20},  # Legend labels and titles
    "text": {"fontSize": 20},  # Text mark size
}

# Bar chart
bars = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('method:N', title=None, axis=alt.Axis(labelAngle=-45), sort = method_order),
    y=alt.Y('mean_diff:Q', title='Mean Difference'),
    color=alt.Color('method:N', title='Evaluator', legend=None, scale=alt.Scale(domain=list(color_mapping.keys()), range=list(color_mapping.values())))
).properties(
    width=120,  # Set the width to 300 pixels
    height=250  # Set the height to 300 pixels
)

# Error bars
error_bars = alt.Chart(chart_data).mark_errorbar().encode(
    alt.X("method:N", sort = method_order),
    alt.Y("ci_lower:Q").title("Mean Difference"),
    alt.Y2("ci_upper:Q"),
    strokeWidth=alt.value(2),
    color=alt.value('gray')
)

# Add value labels
text = bars.mark_text(
    align='center',
    baseline='bottom',
    fontWeight='bold',
    dy=alt.expr(expr=alt.expr.if_(alt.datum.mean_diff >= 0, -1, 20))  # Adjust the position of the text    
).encode(
    text=alt.Text('mean_diff:Q', format='.2f'),
    color=alt.value('black')  # Set text color to black
)

# Combine layers and facet
chart = alt.layer(bars, text, error_bars, data=chart_data).facet(
    column=alt.Column('metric:N', title=None, sort=facet_order),
).configure(**chart_config)  # Apply

# save to html
chart.save("./plots/pls_evaluator_comparison_by_measures.html")

chart



alt.FacetChart(...)

: 